# Постановка задачи.  

У нас есть 5 последовательностей чисел (например, от 0 до 50), кодирующих категориальные признаки.  
Модель должна обучиться на этих последовательностях, чтобы предсказать следующее число в последовательности.  
В модели используется TransformerEncoder, а также добавление позиционных кодировок.

# Как решаем  (на PyTorch)

Создадим класс TransformerEncoder с использованием позиционных кодировок.  
Напишем функцию для обучения модели.  

# Code

## Логика работы кода

### Объяснение кода  
1. PositionalEncoding: Мы создаём класс для позиционных кодировок, которые добавляются к эмбеддингам. Эти кодировки содержат информацию о позиции каждого токена в последовательности, что помогает модели учитывать порядок чисел в последовательности.  

2. TransformerEncoder: Этот класс включает в себя слой эмбеддингов, слой позиционных кодировок, слой TransformerEncoder и полносвязный слой для предсказания.  

3. DataLoader: Создаём обучающий набор данных с 5 последовательностями, где каждой последовательности соответствует целевое значение (следующее число в последовательности).  

4. Обучение: Для обучения мы используем MSELoss как функцию потерь, так как задача — регрессия. Оптимизатор — Adam.  

5. Предсказание: После обучения модели, мы можем подать на неё новую последовательность и получить предсказание следующего числа.  

### Как работает модель  
> Модель обучается на последовательностях чисел, чтобы предсказать следующее значение на основе предыдущих чисел.  

> На каждом шаге обучения модель обновляет свои параметры (включая веса эмбеддингов и параметры self-attention), чтобы улучшить предсказания.  

### Что важно  
> __Позиционные кодировки__ помогают модели понимать порядок чисел в последовательности.  
> __Self-attention__ на основе Transformer позволяет учитывать связи между всеми токенами в последовательности, что важно для понимания контекста.

## Сам код

In [77]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import math

In [66]:
# === Параметры ===
embedding_dim = 32   # размерность эмбеддинга, т. е. размерность вектора, который будет представлять наши последовательности, на одну последовательность один вектор
seq_len = 9         # длина одной последовательности
output_dim = 1       # размерность выходного вектора
batch_size = 5       # 5 числовых последовательностей
num_epochs = 100     # количество эпох обучения
vocab_size = 51      # Словарь, определяющий все возможные значения категорий, # в нашем случае мы выбрали их в виде целых чисел от 0 до 49

1. <font color='lightgreen'>PositionalEncoding:</font> Мы создаём класс для позиционных кодировок, которые добавляются к эмбеддингам. Эти кодировки содержат информацию о позиции каждого токена в последовательности, что помогает модели учитывать порядок чисел в последовательности.

In [78]:
# 1. Класс позиционной кодировки
class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_len=5000):
        super(PositionalEncoding, self).__init__()

        # Генерация позиционных кодировок
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embedding_dim, 2) * -(torch.log(torch.tensor(10000.0))) / embedding_dim)

        pe = torch.zeros(max_len, embedding_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # [1, max_len, embedding_dim]
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## Подробно про __class PositionalEncoding(nn.Module)__  

<font color='lightgreen'>__nn.Module__</font> — это базовый класс любой нейросетевой архитектуры в PyTorch.
Создание своих моделей происходит методом наследования от него.  
 <font color='lightgreen'>**__ init __**</font> - создаём компоненты модели  

1. __Позиции (индексы токенов в последовательности)__  
> <font color='lightgreen'>position = torch.arange(0, max_len).unsqueeze(1)</font>  # shape: [max_len, 1]  
Создаёт тензор с числами от 0 до max_len - 1, по одной позиции в строке.  
>> <font color='lightgreen'>.unsqueeze(1)</font> делает из [max_len] -> [max_len, 1], чтобы можно было делать поблочное умножение.  
<font color='lightgreen'> </font>

2. __Делители (div_term)__  
<font color='lightgreen'>div_term = torch.exp(torch.arange(0, embedding_dim, 2) * -(torch.log(torch.tensor(10000.0)) / embedding_dim))</font>  
Это "частота" для синуса и косинуса. Чем дальше по координате эмбеддинга, тем меньше шаг (частота выше). Каждая чётная координата в эмбеддинге будет иметь свою уникальную частоту.  

3. __Инициализация pe__ — positional encoding матрицы
<font color='lightgreen'>pe = torch.zeros(max_len, embedding_dim) </font>
Матрица, в которую запишутся значения синуса и косинуса.  

4. __Запись синуса и косинуса__  
<font color='lightgreen'>pe[:, 0::2] = torch.sin(position * div_term)  
pe[:, 1::2] = torch.cos(position * div_term)</font>  
Для каждой строки (позиции) мы заполняем:
> - чётные координаты — sin(position * frequency)  
> - нечётные координаты — cos(position * frequency)  
Таким образом, каждое положение в последовательности получает уникальный вектор, но с одинаковой длиной, как и эмбеддинг.  

5. __Добавление размерности батча__  
<font color='lightgreen'>pe = pe.unsqueeze(0)  # shape: [1, max_len, embedding_dim]</font>  
Нам нужно иметь размерность [batch_size, seq_len, embedding_dim].  
Мы делаем unsqueeze(0), чтобы потом можно было сложить с входом x.  

6. __register_buffer__  
<font color='lightgreen'>self.register_buffer('pe', pe)</font>  
pe сохраняется в модели как буфер, а не обучаемый параметр. Он будет сохранён вместе с моделью (state_dict), но не обновляется градиентами.  

7. __Forward: добавление к эмбеддингам__  
<font color='lightgreen'>return x + self.pe[:, :x.size(1)]</font>  
x — это результат embedding(x), размер [batch_size, seq_len, embedding_dim].  
self.pe[:, :x.size(1)] выбирает нужное число позиций (вдруг последовательность короче max_len).  
Далее просто складываем поэлементно эмбеддинг и позиционную кодировку.  


__Итого__  
Мы имеем представление, например, входного вектора в виде матрицы:
где каждая строка — это один токен (или число) в последовательности,
а каждый столбец — это одна координата эмбеддинга этого токена.  

x = [[0.1, 0.2, 0.3],     позиция 0  
     [0.4, 0.5, 0.6]]     позиция 1  

А pe:  
pe = [[0.01, 0.02, 0.03],  позиция 0  
      [0.04, 0.05, 0.06]]  позиция 1  

То результат будет:
x + pe = [[0.11, 0.22, 0.33],
          [0.44, 0.55, 0.66]]
          
<font color='lightgreen'></font>


<font color='lightgreen'> </font>

## далее ...

2. 🔧 Цель класса <font color='lightgreen'>TransformerEncoder</font>  
Класс реализует односторонний (encoder-only) блок трансформера, который:  
- получает на вход последовательность чисел (категорий),  
- превращает каждую категорию в эмбеддинг,  
- добавляет позиционную информацию,  
- пропускает это всё через один или несколько слоёв self-attention,  
- возвращает выход, с которым можно работать: делать классификацию, предсказания и т.д.

In [79]:
# 2. Класс Transformer Encoder для предсказания следующего числа-категории
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super(TransformerEncoder, self).__init__()

        # Вводной слой для эмбеддингов чисел
        self.embedding = nn.Embedding(input_dim, embedding_dim)

        # Позиционная кодировка
        self.pos_encoder = PositionalEncoding(embedding_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # Важно: [batch, seq, dim]
        )

        # Transformer Encoder слой
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Полносвязный слой для предсказания
        self.fc = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        # [x]: Tensor of shape [batch_size, seq_len] — входные индексы категории
        # Применение эмбеддингов
        x = self.embedding(x)                          # [batch_size, seq_len, embedding_dim]
        # Применение позиционной кодировки
        x = self.pos_encoder(x)                          # + позиционная кодировка
        # Применение TransformerEncoder
        output = self.transformer_encoder(x)             # Пропускаем через трансформер

        last_token = output[:, -1, :]       # берём выход с последней позиции
        logits = self.fc(last_token)        # [batch_size, vocab_size]
        return logits

## Подробно про __class TransformerEncoder(nn.Module)__  

 <font color='lightgreen'>__nn.Module__</font> — это базовый класс любой нейросетевой архитектуры в PyTorch.
Создание своих моделей происходит методом наследования от него.  
<font color='lightgreen'>**__ init __**</font>: создаём компоненты модели.  

🔹 <font color='lightgreen'>self.embedding = nn.Embedding(vocab_size, embedding_dim)</font>  
> - Преобразует числа (индексы категорий) во вектора фиксированной длины embedding_dim. Например: [7, 25, 3] → [[0.2, 0.5], [0.1, 0.4], [0.9, 0.7]]  

🔹<font color='lightgreen'>self.pos_encoder = PositionalEncoding(...)</font>  
> - Добавляет информацию о порядке токенов, потому что nn.Embedding сам по себе просто смотрит на значения, но не на их позиции.  

🔹<font color='lightgreen'>encoder_layer = nn.TransformerEncoderLayer(...)  
🔹self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=...)</font>  
> - Один слой self-attention: attention + feedforward + нормализация.  
> - Обернутый в nn.TransformerEncoder он повторяется num_layers раз.

🔹<font color='lightgreen'>def forward(self, x)</font>  
> - <font color='lightgreen'>x = self.embedding(src)</font> Преобразуем индексы в эмбеддинги.  
> - <font color='lightgreen'>x = self.pos_encoder(x)</font> Добавляем позиционную кодировку.  
> - <font color='lightgreen'>output = self.transformer_encoder(x)</font> Пропускаем через трансформер.
> - <font color='lightgreen'>logits = self.fc_out(output[:, -1, :])</font>  Предсказание следующего токена  Мы берём выход последнего токена в последовательности, потому что мы хотим предсказать следующее значение.

🔚 Вывод  
Этот класс подходит для задачи:  
> - "предсказать следующее значение в числовой последовательности".  
> - Учитывает порядок (позиционка).  
> - Гибкий: можно управлять размерностью, числом голов внимания, количеством слоёв.

##  далее

In [80]:
# 3. Данные (пример)
# Генерация случайных данных для обучения
def generate_sequences(num_sequences, seq_len, max_value=50):
    sequences = torch.randint(0, max_value, (num_sequences, seq_len))
    targets = torch.randint(0, max_value, (num_sequences, 1))  # Предсказания для следующего числа
    return sequences, targets

# Генерация 5 последовательностей
sequences, targets = generate_sequences(5, seq_len)

In [81]:
sequences
# targets

tensor([[14, 32,  5, 36, 19, 36, 28, 12, 10],
        [30, 12, 26, 30, 49, 45, 43, 48,  3],
        [15, 29, 13, 18, 46,  7, 27, 15, 23],
        [10, 24,  2, 15, 33, 29, 49, 35, 19],
        [37, 27, 35, 36,  0, 45, 19, 39, 46]])

In [82]:
# 4. Создание DataLoader для обучения
train_data = TensorDataset(sequences, targets)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

In [83]:
# 5. Инициализация модели
input_dim = 51  # Возможные значения от 0 до 50
num_heads = 4  # Количество голов в attention
num_layers = 2  # Количество слоев Transformer
model = TransformerEncoder(input_dim, embedding_dim, num_heads, num_layers, output_dim)

In [84]:
# 6. Настройка оптимизатора и функции потерь
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

Мы выбрали критерий по рассчету ошибок (loss),
loss — это суммарная ошибка классификации по батчу (или по всей выборке, если без батчей).  

📉 2. Каким должен быть нормальный loss?  
Для nn.CrossEntropyLoss():  
- Хороший loss обычно < 1.0 на один пример  
- На старте (если случайные веса) бывает 3–6  
Если loss растёт и не падает — модель не учится  

📌 Возможные причины большого loss:  
Модель не обучается:  
- lr слишком большая/маленькая  
- веса не обновляются  
- плохая инициализация  
- Целевая переменная не подготовлена правильно  
>> например, target имеет форму [batch_size, 1], а должен быть [batch_size] (1D целые индексы)  
- vocab_size не совпадает с количеством классов в fc  
- Проблемы с входами/эмбеддингами, например, входные значения за пределами [0, vocab_size - 1]

In [85]:
# 7. Обучение модели
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        sequences, targets = batch
        optimizer.zero_grad()

        # Прямой проход
        outputs = model(sequences)

        # Вычисление потерь
        # у нас targets имеет форму [5, 1], а надо [5], выпрямляем ()
        loss = criterion(outputs, targets.squeeze())
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(train_loader):.4f}')

Epoch 1/100, Loss: 4.1716
Epoch 2/100, Loss: 4.0831
Epoch 3/100, Loss: 3.9800
Epoch 4/100, Loss: 3.8960
Epoch 5/100, Loss: 3.7136
Epoch 6/100, Loss: 3.7230
Epoch 7/100, Loss: 3.4915
Epoch 8/100, Loss: 3.3803
Epoch 9/100, Loss: 3.3687
Epoch 10/100, Loss: 3.3495
Epoch 11/100, Loss: 3.2239
Epoch 12/100, Loss: 3.0770
Epoch 13/100, Loss: 2.9375
Epoch 14/100, Loss: 2.9691
Epoch 15/100, Loss: 2.8919
Epoch 16/100, Loss: 2.8030
Epoch 17/100, Loss: 2.8398
Epoch 18/100, Loss: 2.6412
Epoch 19/100, Loss: 2.7316
Epoch 20/100, Loss: 2.5016
Epoch 21/100, Loss: 2.4882
Epoch 22/100, Loss: 2.4995
Epoch 23/100, Loss: 2.3309
Epoch 24/100, Loss: 2.4484
Epoch 25/100, Loss: 2.2687
Epoch 26/100, Loss: 2.1979
Epoch 27/100, Loss: 2.0747
Epoch 28/100, Loss: 2.0078
Epoch 29/100, Loss: 1.9878
Epoch 30/100, Loss: 1.8624
Epoch 31/100, Loss: 1.9117
Epoch 32/100, Loss: 1.9135
Epoch 33/100, Loss: 1.8307
Epoch 34/100, Loss: 1.7592
Epoch 35/100, Loss: 1.6182
Epoch 36/100, Loss: 1.6668
Epoch 37/100, Loss: 1.5281
Epoch 38/1

✨     
__Loss-функция__	- nn.CrossEntropyLoss()  
__logits shape__	[- batch_size, vocab_size]  
__target shape__	- [batch_size]  
__target тип__	- LongTensor  
Пример	_loss = criterion(outputs, targets.squeeze())__

In [86]:
# 8. Оценка модели
model.eval()
with torch.no_grad():
    test_seq = torch.randint(0, 50, (1, seq_len))  # Генерация последовательности для предсказания
    prediction_logits = model(test_seq)
print(test_seq) # Сгенерированная случайная последовательность
print(prediction_logits) # Полученные логгиты
print(prediction_logits.shape)



tensor([[42,  8, 31, 42, 41, 35,  0, 17,  7]])
tensor([[-0.5446, -2.2718, -1.5887, -1.7679, -1.5389, -2.4672, -1.7574, -1.9034,
         -2.3494,  2.4925, -2.2023, -1.4525, -2.1091, -1.6808, -1.8527, -2.5895,
         -1.5246, -1.5659,  2.0357, -2.2096,  1.4069, -1.4121, -1.8800, -2.1157,
         -1.1532, -1.7720, -1.7929, -1.2828, -2.4136, -2.7028, -2.3108, -0.6414,
          1.2966, -1.7979, -0.7576, -1.6557, -1.7270, -2.2129, -1.1837, -1.4374,
         -2.2251,  3.0662, -2.8762, -1.2198, -1.7372, -1.7895, -1.6197, -1.8337,
         -1.5302, -2.1711, -1.2917]])
torch.Size([1, 51])


In [87]:
# Полученные вероятности (отнесения предсказанного значения к одному из классов, у нас их - 50, т.е. 50 категорий, размеченных от 0 до 50)
print("_______________________________")
probs = torch.softmax(prediction_logits, dim=-1)
print("Вероятности:\n", probs)
print("_______________________________")
# Посмотреть топ-кандидатов (например, 3-х):

topk_probs, topk_indices = torch.topk(probs, k=3)
print("Топ-3 вероятности:\n", topk_probs)
print("Топ-3 индексы:\n", topk_indices)
print("_______________________________")

# Получить предсказанную категорию

predicted_class = prediction_logits.argmax(dim=-1)
print("Предсказанный класс:\n", predicted_class)

_______________________________
Вероятности:
 tensor([[0.0100, 0.0018, 0.0035, 0.0030, 0.0037, 0.0015, 0.0030, 0.0026, 0.0017,
         0.2092, 0.0019, 0.0040, 0.0021, 0.0032, 0.0027, 0.0013, 0.0038, 0.0036,
         0.1325, 0.0019, 0.0707, 0.0042, 0.0026, 0.0021, 0.0055, 0.0029, 0.0029,
         0.0048, 0.0015, 0.0012, 0.0017, 0.0091, 0.0633, 0.0029, 0.0081, 0.0033,
         0.0031, 0.0019, 0.0053, 0.0041, 0.0019, 0.3713, 0.0010, 0.0051, 0.0030,
         0.0029, 0.0034, 0.0028, 0.0037, 0.0020, 0.0048]])
_______________________________
Топ-3 вероятности:
 tensor([[0.3713, 0.2092, 0.1325]])
Топ-3 индексы:
 tensor([[41,  9, 18]])
_______________________________
Предсказанный класс:
 tensor([41])


In [62]:
print("Logits:", prediction_logits.shape)
print("Target:", targets.shape)
print("Loss:", loss.item())

with torch.no_grad():
    probs = torch.softmax(prediction_logits, dim=-1)
    top_probs, top_indices = probs.topk(1, dim=-1)
    print("Top prob per sample:", top_probs.squeeze().tolist())

Logits: torch.Size([1, 51])
Target: torch.Size([5, 1])
Loss: 934.8015747070312
Top prob per sample: 0.08142051100730896
